In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files:", path)

100%|██████████| 60.3M/60.3M [00:04<00:00, 13.0MB/s]

Extracting files...


Path to dataset files: C:\Users\Admin\.cache\kagglehub\datasets\msambare\fer2013\versions\1


## XÓA dataset trong cache đi cho nhẹ máy


In [ ]:
import kagglehub, shutil

# Xóa toàn bộ thư mục cache của kagglehub
shutil.rmtree(kagglehub.config.get_cache_folder())
print("Đã xóa xong cache của kagglehub!")

### 1. Data preprocessing


In [4]:
import os
train_path = os.path.join(path, 'train')
test_path = os.path.join(path, 'test')


In [45]:
import tensorflow as tf
# 2. Cấu hình các tham số
IMG_SIZE = (48, 48)  # Ảnh FER2013 có kích thước 48x48
BATCH_SIZE = 64

# 3. Load tập train
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="grayscale",  # FER2013 là ảnh xám (1 kênh màu)
    label_mode="categorical", # Trả về dạng one-hot encoding cho 7 nhãn
    shuffle=True
)

# 4. Load tập test (dùng để validation/eval)
val_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="grayscale",
    label_mode="categorical",
    shuffle=False
)

Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.


In [17]:
print("Dataset structure:", train_ds.element_spec)

Dataset structure: (TensorSpec(shape=(None, 48, 48, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 7), dtype=tf.float32, name=None))


In [48]:
for images, labels in train_ds.take(1):
    print("Image shape:", images.shape)
    print("Label shape:", labels.shape)

Image shape: (64, 48, 48, 1)
Label shape: (64, 7)


In [53]:
from collections import Counter
import numpy as np

counter = Counter()

for images, labels in train_ds:
    label_indices = np.argmax(labels.numpy(), axis=1)
    counter.update(label_indices)


for i, class_name in enumerate(train_ds.class_names):
    print(
        f"{class_name}: {counter[i]} images"
    )

angry: 3995 images
disgust: 436 images
fear: 4097 images
happy: 7215 images
neutral: 4965 images
sad: 4830 images
surprise: 3171 images


### Tiến hành Feature engineering


### XỬ LÝ CÁC ĐIỂM ẢNH BẰNG CÁCH ĐIỀU CHỈNH CHÚNG NGẪU NHIÊN

### Xử lý điểm ảnh class Happy

In [54]:
happy_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomContrast(0.05)
])

### Xử lý điểm ảnh class Disgust

In [55]:
disgust_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1)
])

### Xử lý điểm ảnh class còn lại

In [56]:
normal_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])

### Xây dựng mô hình CNN (Mạng nơ ron chập) thường hay sử dụng trong xử lý phân loại ảnh, xử lý âm thanh

In [57]:
model = tf.keras.Sequential([

    data_augmentation,

    tf.keras.layers.Rescaling(1./255),


    # Block 1
    tf.keras.layers.Conv2D(
        32,
        (3,3),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.MaxPooling2D(),


    # Block 2
    tf.keras.layers.Conv2D(
        64,
        (3,3),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.MaxPooling2D(),


    # Block 3
    tf.keras.layers.Conv2D(
        128,
        (3,3),
        padding="same",
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.MaxPooling2D(),


    # Classifier
    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])

### Early stopping tránh dùng quá nhiều CPU GPU

In [58]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

### Chỉ ra mô hình tối ưu trọng số thế nào

In [59]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### THêm reduceOnPlateau

In [60]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [61]:
from tensorflow.keras.callbacks import ModelCheckpoint
# Khai báo callback lưu model tốt nhất
checkpoint_cb = ModelCheckpoint(
    'best_model.keras', 
    save_best_only=True, 
    monitor='val_accuracy', 
    mode='max'
)



### Train dữ liệu

In [62]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=70,
    callbacks=[
        early_stop,
        reduce_lr
    ]
)

Epoch 1/70


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


449/449 ━━━━━━━━━━━━━━━━━━━━ 58s 121ms/step - accuracy: 0.2467 - loss: 1.8785 - val_accuracy: 0.2464 - val_loss: 1.8496 - learning_rate: 1.0000e-04
Epoch 2/70
449/449 ━━━━━━━━━━━━━━━━━━━━ 67s 149ms/step - accuracy: 0.2830 - loss: 1.7725 - val_accuracy: 0.3589 - val_loss: 1.6509 - learning_rate: 1.0000e-04
Epoch 3/70
449/449 ━━━━━━━━━━━━━━━━━━━━ 45s 101ms/step - accuracy: 0.3119 - loss: 1.7230 - val_accuracy: 0.3890 - val_loss: 1.6007 - learning_rate: 1.0000e-04
Epoch 4/70
449/449 ━━━━━━━━━━━━━━━━━━━━ 45s 100ms/step - accuracy: 0.3342 - loss: 1.6770 - val_accuracy: 0.3948 - val_loss: 1.5543 - learning_rate: 1.0000e-04
Epoch 5/70
449/449 ━━━━━━━━━━━━━━━━━━━━ 46s 103ms/step - accuracy: 0.3515 - loss: 1.6422 - val_accuracy: 0.4107 - val_loss: 1.5329 - learning_rate: 1.0000e-04
Epoch 6/70
449/449 ━━━━━━━━━━━━━━━━━━━━ 46s 102ms/step - accuracy: 0.3700 - loss: 1.6116 - val_accuracy: 0.4192 - val_loss: 1.5076 - learning_rate: 1.0000e-04
Epoch 7/70
449/449 ━━━━━━━━━━━━━━━━━━━━ 47s 105ms/step - 

In [3]:
from keras.models import load_model

# 1. Sau khi model.fit(...) chạy xong, LOAD LẠI FILE CHECKPOINT
best_model = load_model('best_model.keras')  # Hoặc 'best_model.h5' tùy tên file bạn đặt

# 2. Đánh giá trên model đã load
test_loss, test_acc = best_model.evaluate(val_ds)
print(f"Best Validation Accuracy: {test_acc}")

NameError: name 'val_ds' is not defined

### Lưu lại mô hình

In [2]:
#Lưu model tốt nhất vào file "emotion_model.keras"
best_model.save("emotion_model.keras")


NameError: name 'best_model' is not defined

### Test mô hình


In [68]:
class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

In [1]:
import tensorflow as tf
import numpy as np

img = tf.keras.utils.load_img(
    "img2.jpg",
    target_size=(48, 48),
    color_mode="grayscale"
)

img_array = tf.keras.utils.img_to_array(img)

# Thêm batch dimension
img_array = tf.expand_dims(
    img_array,
    axis=0
)

# Dự đoán
prediction = model.predict(
    img_array,
    verbose=0
)

predicted_index = np.argmax(
    prediction[0]
)

predicted_emotion = class_names[predicted_index]

confidence = prediction[0][predicted_index]

print("Predicted emotion:", predicted_emotion)
print(f"Confidence: {confidence:.2%}")

NameError: name 'model' is not defined

In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]
